Groundwater | Transport Modeling Track

# Step 1: Model Goal – Defining the Transport Model Problem

> **The 10 steps at a glance:** **1-Goal** → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**The journey:** You already know how to define goals for a flow model. Transport modeling adds a new question – not just *where does the water go?* but *what does it carry along, and where does that end up?*

In your project, that "what" is a **real contaminant** – a solvent, a fuel component, a nitrate or metal plume – that **spills upgradient of a real groundwater heat-exchange (GWHE) doublet** in the Limmat Valley. The doublet's **extraction well doubles as a monitoring / compliance well**, and the sharp question is whether the plume reaches it. Some contaminants travel essentially with the water (a **conservative tracer**); others **sorb** onto the aquifer grains and lag behind, or **decay** along the way. We start from the conservative case – the simplest, most transparent version of the problem – to isolate advection and dispersion, then add the single reaction a real contaminant needs. This notebook builds directly on the calibrated flow model from the flow track.

| **Core content** | ~10 minutes |
|:--|:--|
| **Optional: Exercises & real-world context** | +5 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** clear objectives for a solute-transport model on the Limmat Valley aquifer, starting from a conservative tracer
2. **Frame** the motivating question for a GWHE doublet: does an upgradient contaminant spill **reach the doublet's extraction well (the compliance well)** above the regulatory threshold – or bypass it laterally?
3. **Identify** what additional data and parameters transport modeling requires beyond a flow model
4. **Determine** whether a contaminant is conservative or reactive, and name the minimal reaction – sorption (retardation) or first-order decay – its model must represent
5. **Explain** why a calibrated flow model is the essential foundation for any transport simulation

## Prerequisites

Before starting this notebook, you should have:
- **Completed the Flow Modeling Track** – the transport model is built on top of the calibrated flow model
- Understanding of advection and dispersion concepts from the theory lectures
- Familiarity with the Limmat Valley case study and its wells from the flow track

## Introduction

A flow model tells us where water moves and how fast. A transport model asks the next question: **what travels with the water, and where does it go?**

In your project, that "what" is a **contaminant** that spills into the aquifer upgradient of a GWHE doublet. Contaminants differ in how they travel:

- some move essentially **with the water** – a **conservative tracer**, retardation factor *R* ≈ 1 (e.g. chloride, PFOA);
- some **sorb** onto the aquifer matrix and travel *slower* than the water – retarded, *R* > 1 (e.g. chromium, PCE, atrazine);
- some **decay** or biodegrade en route, losing dissolved mass over time (e.g. benzene, ammonium).

We deliberately **start from the conservative case**: it is the simplest, most transparent transport problem – advection carries the solute with the flow while dispersion and diffusion spread it out – and it isolates exactly the processes the flow field controls. Once you can read a conservative breakthrough curve, a **sorbing or decaying contaminant is a single incremental step**, and the provided model already supports it. *(The physics of sorption and decay comes in [02t](02t_perceptual_model.ipynb) and [03t](03t_modflow_transport.ipynb); here we scope only what the model must represent and what data it needs.)*

**The scenario.** Scattered through the Limmat Valley are **groundwater heat-exchange (GWHE) doublets** – paired wells that pump groundwater up, use it for building heating or cooling, and reinject the used water nearby. Each doublet has an **injection well** (returning clean water) and an **extraction well** that abstracts it. In your project, the **extraction well doubles as a monitoring / compliance well**, and a contaminant spills somewhere upgradient. The doublet's pumping bends the flow field into a **capture zone** around the extraction well, so the plume is either drawn in or drifts past:

> 🎯 **Motivating question:** A contaminant spills upgradient of a real Limmat doublet. Does the plume **reach the extraction (monitoring) well above the regulatory threshold – and if so, when?** Or does it **bypass** the well laterally?

You will answer this by **modifying a provided model** and reading the contaminant breakthrough at the monitoring well.

<details>
<summary><strong>Why a heat-exchange doublet? (the recirculation motivation)</strong></summary>

These are real geothermal installations, and their *own* engineering worry is **recirculation** – reinjected water short-circuiting back to the extraction well and degrading performance. We borrow the **same forced-gradient flow field** for a different question: the extraction-well capture zone that a doublet designs *against* (for its own returned water) is exactly what decides whether an *external* contaminant plume is captured or bypasses. Same wells, same flow field – a contaminant-transport question instead of a thermal one. (The heat-transport packages, MODFLOW 6 **GWE**, are the thermal counterpart of the solute packages, **GWT**, we use here.)
</details>

In this notebook we define the goals and scope of that transport model. We already set the flow model's objectives in the flow track; here we focus on what changes when transport enters the picture:
- **What new questions** can we answer?
- **Where** and at what temporal and spatial scale?
- **What additional data** do we need?
- **Who else** cares about the results?

---

## What Do We Need From This Model?

> 💡 **Our Transport Model Requirements**
>
> We require a **solute-transport model** of a Limmat Valley GWHE doublet to:
>
> 1. **Track a contaminant** from an upgradient spill as it is advected and dispersed through the doublet's flow field
> 2. **Quantify breakthrough at the monitoring (extraction) well** – whether the plume is captured or bypasses, its peak concentration versus the regulatory threshold, and the first-exceedance time
> 3. Build on our existing **calibrated flow model** using **open-source tools** (MODFLOW 6 GWT)
> 4. **Represent the one reaction a real contaminant needs** – linear sorption (retardation) *or* first-order decay – when the solute is not conservative (the GWT **MST** package already provides both)
>
> The model should be **fast enough for interactive exploration** (each breakthrough simulation completes in ≤10 minutes).

---

## 🎯 Mission Objectives – What Are We Solving?

The flow model answered: *where does the water go and how much?* Transport modeling opens up a different set of questions.

### Primary Objectives — Contaminant Transport to the Monitoring Well

Our Limmat Valley doublet transport model should be able to:

1. **Simulate contaminant migration** from an upgradient spill through the doublet's flow field
2. **Predict breakthrough (arrival time and peak concentration)** at the monitoring (extraction) well, versus the regulatory threshold – the capture-vs-bypass question
3. **Delineate the capture zone** of the extraction well and relate it to the spill's position (captured or bypassed)
4. **Test sensitivity** to the transport parameters we are least sure of – dispersivity and effective porosity

**Why a conservative tracer first?** It isolates the transport processes that the flow field controls – advection and dispersion – without the added uncertainty of reactions. Once you can read a conservative breakthrough curve, sorption and decay are incremental additions.

### Educational Objectives

As a teaching tool, the transport model should also:

- Illustrate how the **flow field controls transport** (the doublet's forced gradient decides whether the plume is captured)
- Show the roles of **advection, dispersion, and diffusion**
- Demonstrate the difference between **conservative and reactive** transport
- Highlight the importance of **numerical stability** (Peclet and Courant criteria)

<details>
<summary><strong>Optional: Exercise – Reasoning about capture vs bypass</strong></summary>

> ✏️ **Exercise: Will the spill reach the well?**
>
> Before we model anything, reason qualitatively about the motivating question:
>
> 1. The extraction well draws a **capture zone** around itself. Describe two paths the contaminant could take from the spill – one that is captured by the extraction (monitoring) well, and one that escapes past it downgradient. What controls which one dominates?
> 2. Dispersivity spreads the plume out around the average flow path. Could *increasing* dispersivity change the verdict at the monitoring well? Explain why it can cut both ways – a wider plume is more likely to clip the well, but its peak concentration is lower.
> 3. Effective porosity sets the seepage velocity ($v = q / n_e$). If you halved the effective porosity, what would happen to the plume's arrival time at the monitoring well?
>
> **Hint:** Advection follows the doublet's forced-gradient flow field; the extraction well's capture-zone half-width versus the spill's lateral offset decides capture vs bypass. Arrival *time* scales with travel distance divided by seepage velocity.

<details>
<summary><strong>Click to check your reasoning</strong></summary>

**1. Capture vs bypass.**
- *Captured:* a spill released within the extraction well's **capture zone** follows the doublet's forced-gradient streamlines that converge on the well, so it is intercepted and registers in the breakthrough.
- *Bypass:* a spill whose **lateral offset exceeds the capture-zone half-width** rides the regional streamlines past the well and escapes downgradient – little or no breakthrough.
- *What controls it:* the spill's lateral offset versus the capture-zone half-width, screened with $y_{\max} = Q/(2Ti)$ – a larger pumping rate $Q$ widens the zone, a steeper regional gradient $i$ narrows it.
- *Caveat:* $y_{\max}$ is the **single-well** estimate. In a doublet the injection well can both **narrow the effective capture** (part of what the well pulls is re-injected clean water recirculating back, not upstream flow) and **deflect the plume sideways** (the injection mound). Treat $y_{\max}$ as a first-order screen, not the last word.
- Capture only answers *whether* the plume reaches the well; *when* it arrives is set by travel distance and seepage velocity – see Q3.

**2. Increasing dispersivity cuts both ways.**
Dispersion mainly **reshapes and dilutes** the plume rather than redirecting it:
- the **transverse** term ($\alpha_T$) widens the plume laterally, so a plume whose centerline would just bypass can **clip** the well at low concentration (a marginal bypass becomes a small detection);
- the **longitudinal** term ($\alpha_L$) brings the **toe** in earlier and stretches the tail;
- but all of this spreading **lowers the peak**, so an on-centerline plume can be **diluted below the threshold**.

So more dispersivity can flip a *borderline* verdict either way; for a clearly captured or clearly bypassing plume it changes only shape and timing. This is the key distinction for a compliance well: **reaching the well is not the same as exceeding the threshold** – a plume can arrive yet stay compliant through dilution. (In this course $\alpha_L = 10$ m and $\alpha_T = 1$ m are fixed, so this is a thought experiment – but it is the transverse term that governs the lateral clip and the longitudinal term that governs early arrival and peak dilution.)

**3. Halving effective porosity.**
Seepage velocity is $v = q/n_e$. On a **steady** flow field the specific discharge $q$ and the streamlines are fixed by the head solution – independent of porosity – so halving $n_e$ **doubles** the velocity and the advective arrival time is **halved exactly** (only the dispersive spread of the front carries any "≈").
- *For a sorbing solute the speed-up is weaker than 2×.* The transport velocity is $v_R = q/(n_e + \rho_b K_d)$, and the retardation factor $R = 1 + \rho_b K_d/n_e$ **itself contains** $n_e$. So both terms respond to porosity: halving $n_e$ shortens the arrival time by a factor between 2 (non-sorbing) and 1 (strongly sorbing), approaching *no change* for a strongly retarded solute.

</details>

</details>

---

## 🗺️ Setting the Scene – Where and When?

We use the same Limmat Valley domain and the same flow grid as the flow model, but transport places different demands on resolution and time.

### Spatial Scale

| Dimension | Value | Rationale |
|-----------|-------|-----------|
| **Domain extent** | Same as flow model | Full valley, focused on the doublet and the spill → well corridor |
| **Grid resolution** | Inherited flow grid (~50 m cells), locally refined along the spill → well corridor | Sharp concentration gradients need finer cells than flow — see the note below |
| **Vertical layers** | 1 | Single layer — prioritises fast run times for interactive exploration |

> ⚠️ **A grid that is fine enough for flow is often too coarse for transport.** On the ~50 m flow grid, the plume front is artificially smeared (the *grid Peclet* number is too high). The provided model is therefore **refined along the spill → well corridor** so the breakthrough is trustworthy. You will see exactly how this is diagnosed and fixed in **[04t](04t_model_implementation.ipynb)** — here it is enough to know that resolution must be checked against the *transport*, not just the flow.

### Temporal Scale

| Aspect | Choice | Rationale |
|--------|--------|-----------|
| **Simulation type** | Transient | The plume evolves over time, even on a steady flow field |
| **Time step** | Small near the wells | Must satisfy the Courant criterion where velocities are highest (near the doublet) |
| **Simulation period** | Long enough for breakthrough | Run until the plume either reaches the monitoring well or clearly bypasses it |

> **Key insight:** Transport simulations are almost always **transient** – even when the underlying flow is steady-state. The concentration at the monitoring well after 1 month looks very different from after 2 years.

---

## Essential Data – What Do We Need?

The flow model already supplies the velocity field. Solute transport adds its own, shorter, list of requirements.

### Data for Solute Transport

| Category | Data Type | Source | Availability |
|----------|-----------|--------|--------------|
| **Porosity** | Effective porosity *n*<sub>e</sub> | Literature, lab tests | ⚠️ Estimated |
| **Dispersivity** | Longitudinal *α*<sub>L</sub>, transverse *α*<sub>T</sub> | Tracer tests, literature | ⚠️ Uncertain |
| **Diffusion** | Molecular diffusion *D*<sub>m</sub> | Literature | Good |
| **Source term** | Spill location, concentration, release schedule (pulse/continuous) | Scenario definition | Known |
| **Background** | Ambient (initial) concentration | Definition | Set to zero |
| **Receptor** | Monitoring (extraction) well location and rate; regulatory threshold | Doublet concession | Known |
| **Sorption** *(reactive contaminants only)* | Distribution coefficient *K*<sub>d</sub>, bulk density *ρ*<sub>b</sub> | Literature, lab | ⚠️ Uncertain |
| **Decay** *(reactive contaminants only)* | First-order decay constant *λ* or half-life *t*<sub>½</sub> | Literature | ⚠️ Uncertain |

> **Working defaults used in this track** (gravel aquifer): effective porosity *n*<sub>e</sub> = 0.20; longitudinal dispersivity *α*<sub>L</sub> = 10 m; transverse dispersivity *α*<sub>T</sub> = 1 m (= *α*<sub>L</sub>/10); molecular diffusion *D*<sub>m</sub> ≈ 8.6 × 10⁻⁵ m²/d (≈ 1 × 10⁻⁹ m²/s). You will revisit and stress-test *α*<sub>L</sub> as a sensitivity exercise later in the track.

> **If your contaminant reacts** (5 of the 9 case-study contaminants do): a **sorbing** solute needs a distribution coefficient *K*<sub>d</sub> plus the aquifer **bulk density** *ρ*<sub>b</sub> (together they set the retardation *R*); a **decaying** solute needs a first-order decay constant *λ* (or half-life). These are **pinned per group** in your `case_config_transport.yaml` and the provided model reads them automatically — none of the case studies needs both at once.

> **Key terms:**
> - *Effective porosity (n<sub>e</sub>)*: The fraction of pore space through which water (and the solute) actually flows. Controls transport velocity via *v* = *q* / *n*<sub>e</sub>.
> - *Dispersivity (α)*: A length scale that governs mechanical mixing. Notoriously scale-dependent – values measured in lab columns don't apply at field scale.
> - *Conservative tracer*: A substance that moves with the water without sorbing, decaying, or reacting (retardation factor *R* ≈ 1).
> - *Retardation factor (R)*: How much slower a sorbing solute travels than the water – *R* = 1 + *ρ*<sub>b</sub>*K*<sub>d</sub> / *n*<sub>e</sub>; *R* ≈ 1 is conservative, *R* > 1 sorbs (quantified in [03t](03t_modflow_transport.ipynb)).
> - *First-order decay*: Loss of dissolved mass at a rate proportional to concentration, set by a decay constant *λ* (half-life *t*<sub>½</sub> = ln 2 / *λ*) – e.g. biodegradation of benzene.

### Data Quality Considerations

- **The source term often dominates the uncertainty**: what is released, at what concentration and over what duration. In this teaching study the spill is *defined by the scenario* (location, concentration, release schedule), which is what makes the breakthrough question well-posed.
- **Dispersivity is scale-dependent**: it grows with the distance the plume travels, because longer paths sample more heterogeneity. A lab value of ~1 cm can become tens of metres at field scale – one of the most uncertain parameters in solute transport.
- **Effective porosity is rarely measured directly** and sets the seepage velocity, so it directly controls arrival time.
- **Reaction parameters are uncertain too**: sorption *K*<sub>d</sub> and decay rates are compound- and site-specific, often taken from literature ranges rather than measured on your aquifer.
- **Observations**: real concentration data are typically sparse and expensive; in this teaching study the "observation" is the simulated breakthrough at the monitoring well.

### Comprehension Check

Before moving on, make sure you can answer these:

> **1. For a contaminant-spill study near a GWHE doublet, which two inputs usually carry the most uncertainty – and why?**
>
> **2. Why does this course have you *modify a provided* transport model rather than build one from scratch?**

<details>
<summary>Click to check your answer</summary>

**1. The source term and dispersivity.**
- The **source term** (what is released, at what concentration and over what duration) is the dominant uncertainty in most real solute studies – although here the spill is fixed *by the scenario*, which is what makes the breakthrough question well-posed.
- **Dispersivity** is *scale-dependent*: it grows with travel distance because longer paths sample more heterogeneity, so a lab value of ~1 cm can become tens of metres in the field.

Effective porosity is a close third, because it sets the seepage velocity and hence the arrival time. These are the parameters you will deliberately stress-test, because the flow field – and therefore advection – is already constrained by the calibrated flow model.

**2. Because building and refining a transport grid correctly is instructor-time work.** A grid that is adequate for flow is *not* adequate for transport (the grid-Peclet problem, shown in 04t), and refining it reliably along the spill → well corridor is fiddly. Handing you a pre-built, pre-refined model lets you spend your time on the modeling *decisions and judgment* – source, dispersivity, scenario, and interpreting the breakthrough – which is where the learning is.
</details>

---

## 👥 Key Players – Who Cares About the Results?

Transport results often carry higher stakes than flow-only results – contamination affects human health, and well-field operators care whether their reinjected water comes back.

### Primary Users

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Students** | Learning transport concepts | Clear breakthrough curves, interactive scenarios |
| **Instructors** | Teaching advection–dispersion | An illustrative, fast-running doublet example |

<details>
<summary><strong>Optional: Real-world stakeholders</strong></summary>

### In a Real-World Context

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Regulators** | Compliance with concentration limits | Conservative predictions, uncertainty bounds |
| **Site owners** | Contamination liability, remediation costs | Plume delineation, arrival-time estimates |
| **Water utilities** | Wellhead protection, safe yield | Capture-zone analysis, early warning |
| **GWHE / geothermal operators** | System efficiency, avoiding recirculation | Whether and how fast reinjected water short-circuits to the extraction well |

</details>

---

## Summary: Transport Model Goal Definition

> **🎯 Mission**
>
> **Track a contaminant** from an upgradient spill through a real Limmat GWHE doublet's flow field and **read its breakthrough at the monitoring (extraction) well** – is it captured or does it bypass, does it exceed the regulatory threshold, and when?
> - Primary: arrival time and peak concentration at the monitoring well vs threshold; the extraction-well capture zone
> - Sensitivity: dispersivity (*α*<sub>L</sub>) and effective porosity
>
> **🗺️ Setting the Scene**
>
> - Spatial: same domain and flow grid, locally refined along the spill → well corridor (a flow-adequate grid is too coarse for transport – see 04t)
> - Temporal: transient simulation on a steady flow field, run until breakthrough or clear bypass
>
> **📊 Essential Intel**
>
> - Foundation: the calibrated flow model (supplies the velocity field)
> - Add: effective porosity, dispersivity (*α*<sub>L</sub> = 10 m, *α*<sub>T</sub> = 1 m), and the spill source term – plus, for a reactive contaminant, **one reaction parameter** (a sorption *K*<sub>d</sub> or a decay rate *λ*)
>
> **🔑 Key Message**
>
> A conservative tracer isolates the transport physics the flow field controls; the source term and dispersivity carry the uncertainty. A sorbing or decaying contaminant adds just one reaction parameter on top.
>
> **👥 Key Players**
>
> - Primary: Students and instructors (educational use)
> - Real-world: Regulators, site owners, water utilities, GWHE/geothermal operators

---

## What You're Taking Forward

You now have a clear, well-posed transport goal: a **contaminant spill upgradient of a GWHE doublet**, with the **breakthrough at the monitoring (extraction) well** – captured or bypassed, above threshold or not – as the quantity of interest. The calibrated flow model supplies the velocity field; transport adds three things – **effective porosity, dispersivity, and the spill source term** – plus, when your contaminant is reactive, **one reaction parameter** (a sorption *K*<sub>d</sub> or a decay rate).

Remember the two framing decisions:
- You will **modify a provided, pre-refined model**, not build or refine one.
- The biggest uncertainties are the **source term and dispersivity** – which is exactly where your judgment and sensitivity testing belong.

---

## Next Steps

With the transport goal defined, we extend our perceptual model to include transport processes – what actually drives a contaminant's movement from the spill toward the doublet in the Limmat Valley?

**Continue to:** [02t_perceptual_model.ipynb](02t_perceptual_model.ipynb)